# Policy learning: compare two work clocks

Day 21 · Chapter 12. CPU, authored synthetic data, no downloads. Predict before each reference, change one declared variable, and interpret the result. Code is complete so the notebook runs from beginning to end. Completing its execution is material verification, not an assessment of learner mastery.

[Canonical chapter](../../book/chapters/12-language-generation-as-a-policy.md)


In [ ]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
while not (root / "src" / "dongxi_llms").is_dir():
    if root.parent == root:
        raise RuntimeError("Open this notebook within the Dongxi_LLMs repository")
    root = root.parent
sys.path.insert(0, str(root / "src"))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white",
                     "font.family": "DejaVu Sans", "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
blue, orange, green = "#356a9b", "#b97139", "#3d8268"



### Prediction

## Pre-run prediction

Six prompts ask (a+b) mod 3. Each has separate logits, so this experiment studies optimization, not unseen arithmetic reasoning. Four answers per prompt are sampled each rollout update. PPO takes three gradient epochs; the other methods take one. Predict how that affects sample-efficiency versus compute-efficiency comparisons.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
from dongxi_llms.policy_gradient_lab import procedural_policy
modes=("reinforce","baseline","rloo","ppo")
seeds=(1921,1922,1923)
results={mode:[procedural_policy(mode,seed) for seed in seeds] for mode in modes}
fig,axes=plt.subplots(1,2,figsize=(12,4))
for mode,rows in results.items():
    curves=torch.tensor([r["history"] for r in rows])
    mean=curves.mean(0);low=curves.min(0).values;high=curves.max(0).values
    rollout=torch.arange(1,len(mean)+1)
    axes[0].plot(rollout,mean,label=mode);axes[0].fill_between(rollout,low,high,alpha=.12)
    passes=rollout*(3 if mode=="ppo" else 1)
    axes[1].plot(passes,mean,label=mode)
axes[0].set(xlabel="rollout updates (24 samples each)",ylabel="exact success probability",ylim=(0,1))
axes[1].set(xlabel="gradient passes",ylabel="exact success probability",ylim=(0,1))
axes[0].legend();axes[1].legend();plt.tight_layout();plt.show()


### Reference explanation

**Read the figure:** left aligns sampled-completion budget; right exposes optimizer-work differences. Shading is the min/max of three seeds, not a confidence interval. PPO's oracle pre-rollout value deliberately removes critic error. It must not be presented as a learned-value result.


![Saved reference plot for 01 policy algorithm comparison, figure 1](../figures/chapter-12/day-21-01_policy_algorithm_comparison-01.png)

Saved reference output from the verified CPU run. Run the preceding cell to regenerate the current experiment; this image does not change when parameters are edited.


### Prediction

## Exercise 2: report the contract

Which algorithm has more gradient work under the same 3,840 sampled completions? Does a higher score here establish a better LLM algorithm?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
for mode,rows in results.items():
    print(mode,{"seed_success":[r["success_probability"] for r in rows],
                "sampled_completions":rows[0]["sampled_completions"],
                "gradient_steps":rows[0]["gradient_steps"],
                "rollout_updates":rows[0]["rollout_updates"]})
assert all(r["sampled_completions"]==3840 for rows in results.values() for r in rows)
assert results["ppo"][0]["gradient_steps"]==480
assert results["rloo"][0]["gradient_steps"]==160


### Reference explanation

**Solution:** the task and sampling budget are matched; PPO reuses each batch and takes 480 rather than 160 gradient passes. There is no unseen prompt split because separate logits memorize each context's best action. Conclusions concern the declared finite task and this budget.


### Prediction

## Exercise 3: connect the learning methods

Explain what supplied each target: an external demonstration, a recorded pair judgment, or a sampled outcome. Then change G from 4 to 2 in one method and report the new sample budget, rather than silently calling it the same comparison.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
changed=procedural_policy("rloo",seed=1921,group=2)
print({"G2_samples":changed["sampled_completions"],
       "G2_success":changed["success_probability"],
       "G4_success":results["rloo"][0]["success_probability"]})
contracts={"SFT":"externally demonstrated targets",
           "DPO":"recorded preference pairs and fixed reference",
           "REINFORCE":"current-policy samples weighted by outcome reward",
           "RLOO":"same samples with other-completion baseline",
           "PPO":"old-policy rollouts, detached advantage, clipped surrogate"}
contracts


### Reference explanation

**Solution:** methods can share likelihood-scoring code while optimizing different distributions and objectives. G changes sample cost and the baseline's noise. A model-scale extension must additionally measure rollout throughput, policy/reference memory, verifier validity, retained capabilities, and optimizer health. The report preserves all declared seeds and negative comparisons.
